# 🔬 Corneal Ulcer Segmentation — U-Net++ with ResNet-50

**Dataset:** SUSTech-SYSU (712 fluorescein staining images + ulcer masks)  
**Model:** U-Net++ with ResNet-50 encoder (pretrained on ImageNet)  
**Strategy:** Phase 1 (encoder frozen) → Phase 2 (fine-tuning)  
**Loss:** Combined Dice + BCE  
**Extras:** Grad-CAM, all metrics, model export for FastAPI

---

## 📁 Step 1 — Mount Google Drive & Set Save Directory

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/corneal_ulcer_unetpp"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Save directory ready: {SAVE_DIR}")

ModuleNotFoundError: No module named 'google.colab'

## 📦 Step 2 — Install Dependencies

In [3]:
!pip install segmentation-models-pytorch albumentations -q
print("✅ Packages installed")

✅ Packages installed


## 🗂️ Step 3 — Clone Dataset from GitHub

In [4]:
import os

DATASET_DIR = "/content/CornealUlcer"

if not os.path.exists(DATASET_DIR):
    !git clone https://github.com/VaradSinghal/CornealUlcer.git /content/CornealUlcer
    print("✅ Dataset cloned")
else:
    print("✅ Dataset already exists")

# Verify folder structure
for folder in ["rawImages", "ulcerLabels", "corneaLabels"]:
    path = os.path.join(DATASET_DIR, folder)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {folder}/  →  {count} files")

✅ Dataset cloned
  rawImages/  →  712 files
  ulcerLabels/  →  354 files
  corneaLabels/  →  712 files


Cloning into '/content/CornealUlcer'...
Updating files:  12% (868/6889)
Updating files:  13% (896/6889)
Updating files:  14% (965/6889)
Updating files:  15% (1034/6889)
Updating files:  15% (1074/6889)
Updating files:  16% (1103/6889)
Updating files:  17% (1172/6889)
Updating files:  18% (1241/6889)
Updating files:  18% (1274/6889)
Updating files:  19% (1309/6889)
Updating files:  20% (1378/6889)
Updating files:  21% (1447/6889)
Updating files:  22% (1516/6889)
Updating files:  22% (1537/6889)
Updating files:  23% (1585/6889)
Updating files:  24% (1654/6889)
Updating files:  25% (1723/6889)
Updating files:  26% (1792/6889)
Updating files:  27% (1861/6889)
Updating files:  27% (1923/6889)
Updating files:  28% (1929/6889)
Updating files:  29% (1998/6889)
Updating files:  30% (2067/6889)
Updating files:  31% (2136/6889)
Updating files:  32% (2205/6889)
Updating files:  33% (2274/6889)
Updating files:  34% (2343/6889)
Updating files:  35% (2412/6889)
Updating files:  36% (2481/6889)
Updati

## 🔧 Step 4 — Imports

In [5]:
import os
import json
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Device ────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"    GPU: {torch.cuda.get_device_name(0)}")

# ── Reproducibility ───────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
print("✅ Seed set for reproducibility")

🖥️  Using device: cpu
✅ Seed set for reproducibility


## ⚙️ Step 5 — Hyperparameter Config

All hyperparameters in one place — easy to change for viva/demo.

In [6]:
# ── Paths ─────────────────────────────────────────────────────────
RAW_DIR    = "/content/CornealUlcer/rawImages"
MASK_DIR   = "/content/CornealUlcer/ulcerLabels"

# ── Image ─────────────────────────────────────────────────────────
IMG_SIZE   = 256          # Both width and height

# ── Training - Phase 1 (Encoder Frozen) ───────────────────────────
EPOCHS_P1  = 10           # Epochs with encoder frozen
LR_P1      = 1e-4         # Learning rate for Phase 1
BATCH_SIZE = 16           # Batch size (16 safer for Colab T4)
PATIENCE   = 3            # Early stopping patience

# ── Training - Phase 2 (Fine-tuning) ──────────────────────────────
EPOCHS_P2  = 8            # Extra fine-tuning epochs
LR_P2      = 1e-5         # Very low LR to avoid destroying pretrained weights

# ── Model ─────────────────────────────────────────────────────────
ENCODER         = "resnet50"
ENCODER_WEIGHTS = "imagenet"
NUM_WORKERS     = 2

print("✅ Configuration ready")
print(f"   Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   Phase 1     : {EPOCHS_P1} epochs @ lr={LR_P1}")
print(f"   Phase 2     : {EPOCHS_P2} epochs @ lr={LR_P2}")
print(f"   Encoder     : {ENCODER} pretrained on {ENCODER_WEIGHTS}")

✅ Configuration ready
   Image size  : 256x256
   Batch size  : 16
   Phase 1     : 10 epochs @ lr=0.0001
   Phase 2     : 8 epochs @ lr=1e-05
   Encoder     : resnet50 pretrained on imagenet


## 📂 Step 6 — Build Dataset File Lists

In [1]:
mask_files = sorted(os.listdir(MASK_DIR))

image_paths, mask_paths = [], []

for mask_file in mask_files:
    stem     = os.path.splitext(mask_file)[0]   # e.g. "396"
    img_path = os.path.join(RAW_DIR,  stem + ".jpg")
    msk_path = os.path.join(MASK_DIR, mask_file)

    if os.path.exists(img_path):
        image_paths.append(img_path)
        mask_paths.append(msk_path)

print(f"✅ Total matched image-mask pairs: {len(image_paths)}")
print(f"   Sample image : {image_paths[0]}")
print(f"   Sample mask  : {mask_paths[0]}")

# ── 70 / 15 / 15 split ────────────────────────────────────────────
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    image_paths, mask_paths, test_size=0.30, random_state=42
)
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs, temp_masks, test_size=0.50, random_state=42
)

print(f"\n🗂️  Dataset split:")
print(f"   Train : {len(train_imgs):>4}  ({len(train_imgs)/len(image_paths)*100:.1f}%)")
print(f"   Val   : {len(val_imgs):>4}  ({len(val_imgs)/len(image_paths)*100:.1f}%)")
print(f"   Test  : {len(test_imgs):>4}  ({len(test_imgs)/len(image_paths)*100:.1f}%)")

NameError: name 'os' is not defined

## 🎨 Step 7 — Augmentation Pipeline

- **Train**: geometric + photometric augmentations (handles small dataset, fluorescein images)
- **Val/Test**: only resize + normalize (no augmentation, for fair evaluation)

In [ ]:
# ImageNet stats — matches ResNet-50 pretraining
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Training transforms (WITH augmentation) ───────────────────────
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    # Geometric — eye images: horizontal flip OK (left/right eye symmetry),
    # no vertical flip (eyes don't appear upside down clinically)
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),                         # ±20° for head tilt
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.1,
        rotate_limit=0, p=0.4
    ),

    # Photometric — handles varying fluorescein concentration & lighting
    A.RandomBrightnessContrast(
        brightness_limit=0.2, contrast_limit=0.2, p=0.5
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),          # Simulates focus variation
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),       # Sensor noise
    A.CLAHE(clip_limit=2.0, p=0.3),                    # Enhances local contrast

    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
], additional_targets={"mask": "mask"})

# ── Val / Test transforms (NO augmentation) ───────────────────────
eval_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
], additional_targets={"mask": "mask"})

print("✅ Transforms defined")
print(f"   Train: resize → geometric aug → photometric aug → normalize → tensor")
print(f"   Val  : resize → normalize → tensor")

## 🏗️ Step 8 — Dataset Class & DataLoaders

In [ ]:
class CornealUlcerDataset(Dataset):
    """
    Loads fluorescein eye images and binary ulcer masks.
    Images : RGB .jpg
    Masks  : Grayscale .png  →  binarized to 0/1 float
    """
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image as RGB numpy array
        image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))

        # Load mask as grayscale, binarize (threshold 127)
        mask  = np.array(Image.open(self.mask_paths[idx]).convert("L"))
        mask  = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]

        # mask shape: (H, W) → (1, H, W) for binary segmentation
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.tensor(mask).unsqueeze(0)

        return image, mask


# ── Datasets ──────────────────────────────────────────────────────
train_dataset = CornealUlcerDataset(train_imgs, train_masks, transform=train_transform)
val_dataset   = CornealUlcerDataset(val_imgs,   val_masks,   transform=eval_transform)
test_dataset  = CornealUlcerDataset(test_imgs,  test_masks,  transform=eval_transform)

# ── DataLoaders ───────────────────────────────────────────────────
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print("✅ Datasets and DataLoaders ready")
print(f"   Train  : {len(train_dataset):>4} samples  ({len(train_loader)} batches)")
print(f"   Val    : {len(val_dataset):>4} samples  ({len(val_loader)} batches)")
print(f"   Test   : {len(test_dataset):>4} samples  ({len(test_loader)} batches)")

# ── Quick sanity check ────────────────────────────────────────────
sample_img, sample_mask = train_dataset[0]
print(f"\n   Image tensor shape : {sample_img.shape}   dtype: {sample_img.dtype}")
print(f"   Mask  tensor shape : {sample_mask.shape}   dtype: {sample_mask.dtype}")
print(f"   Unique mask values : {sample_mask.unique().tolist()}")

## 👁️ Step 9 — Visualize Sample Images

In [ ]:
def denormalize(tensor):
    """Reverse ImageNet normalization for visualization."""
    mean = np.array(IMAGENET_MEAN)
    std  = np.array(IMAGENET_STD)
    img  = tensor.cpu().permute(1, 2, 0).numpy()
    img  = img * std + mean
    return np.clip(img, 0, 1)


fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle("Sample Fluorescein Images with Ulcer Masks", fontsize=14, fontweight="bold")

for i in range(3):
    img, mask = train_dataset[i * 15]
    img_display  = denormalize(img)
    mask_display = mask.squeeze().numpy()

    # Raw image
    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f"Sample {i+1} — Raw Image", fontsize=10)
    axes[i, 0].axis("off")

    # Mask
    axes[i, 1].imshow(mask_display, cmap="gray")
    axes[i, 1].set_title(f"Ulcer Mask  ({int(mask_display.sum())} px)", fontsize=10)
    axes[i, 1].axis("off")

    # Overlay (mask in red on image)
    overlay = img_display.copy()
    overlay[:, :, 0] = np.where(mask_display > 0.5, 1.0, overlay[:, :, 0])
    overlay[:, :, 1] = np.where(mask_display > 0.5, 0.0, overlay[:, :, 1])
    overlay[:, :, 2] = np.where(mask_display > 0.5, 0.0, overlay[:, :, 2])
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title("Overlay (red = ulcer)", fontsize=10)
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Sample visualization saved")

## 🧠 Step 10 — Build U-Net++ with ResNet-50

**Architecture:**
- Encoder: ResNet-50 (pretrained on ImageNet) — 5 stages, 2048 channels at bottleneck
- Decoder: U-Net++ (nested dense skip connections — better than plain U-Net)
- Output: 256×256×1 binary segmentation mask

**Why U-Net++ over U-Net?**  
U-Net++ uses nested skip connections that bridge the semantic gap between encoder/decoder features. Captures both fine boundaries and global context.

In [ ]:
model = smp.UnetPlusPlus(
    encoder_name    = ENCODER,          # resnet50
    encoder_weights = ENCODER_WEIGHTS,  # imagenet pretrained
    in_channels     = 3,                # RGB input
    classes         = 1,                # Binary segmentation
    activation      = None              # Raw logits — we apply sigmoid in loss/eval
).to(DEVICE)

# ── Count parameters ──────────────────────────────────────────────
total_params    = sum(p.numel() for p in model.parameters())
encoder_params  = sum(p.numel() for p in model.encoder.parameters())
decoder_params  = total_params - encoder_params

print("✅ U-Net++ with ResNet-50 built")
print(f"   Total parameters   : {total_params:>12,}")
print(f"   Encoder (ResNet-50): {encoder_params:>12,}  (will be frozen in Phase 1)")
print(f"   Decoder (U-Net++)  : {decoder_params:>12,}  (always trainable)")

## 📉 Step 11 — Loss Function & Optimizer

**Combined Loss = 0.5 × Dice + 0.5 × BCE**
- Dice: handles class imbalance (ulcer pixels << background pixels)
- BCE: stable gradient signal pixel-by-pixel
- Together: best of both for small, irregular regions

In [ ]:
dice_loss_fn = smp.losses.DiceLoss(mode="binary", smooth=1e-6)
bce_loss_fn  = smp.losses.SoftBCEWithLogitsLoss()

def combined_loss(preds, targets):
    """0.5 * Dice + 0.5 * BCE"""
    return 0.5 * dice_loss_fn(preds, targets) + 0.5 * bce_loss_fn(preds, targets)


def compute_metrics_batch(preds_logits, masks):
    """
    Computes Dice and IoU for a batch.
    preds_logits: raw model output (before sigmoid)
    masks       : ground truth (0/1 float)
    """
    preds_bin = (torch.sigmoid(preds_logits) > 0.5).float()
    smooth    = 1e-6
    inter     = (preds_bin * masks).sum()
    union     = preds_bin.sum() + masks.sum()
    dice      = (2 * inter + smooth) / (union + smooth)
    iou       = (inter + smooth) / (union - inter + smooth)
    return dice.item(), iou.item()


print("✅ Loss function: 0.5 × DiceLoss + 0.5 × SoftBCE")
print("✅ Metrics function: Dice coefficient + IoU")

## 🔄 Step 12 — Training Helper Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """Run one training epoch. Returns avg loss."""
    model.train()
    losses = []
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss  = combined_loss(preds, masks)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def validate(model, loader, device):
    """Run validation. Returns avg loss, avg dice, avg iou."""
    model.eval()
    losses, dices, ious = [], [], []
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            loss  = combined_loss(preds, masks)
            dice, iou = compute_metrics_batch(preds, masks)
            losses.append(loss.item())
            dices.append(dice)
            ious.append(iou)
    return float(np.mean(losses)), float(np.mean(dices)), float(np.mean(ious))


print("✅ Training and validation helper functions ready")

## 🏋️ Step 13 — Phase 1: Train with Encoder Frozen

**Strategy:** Freeze all ResNet-50 encoder parameters.  
Only the U-Net++ decoder (5M params) trains.  
**Why:** Prevents random decoder weights from destroying pretrained ImageNet features during early training.

In [ ]:
# ── Freeze encoder ────────────────────────────────────────────────
for param in model.encoder.parameters():
    param.requires_grad = False

trainable_p1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 1: Encoder FROZEN")
print(f"  Trainable parameters: {trainable_p1:,}  (decoder only)")

optimizer_p1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P1, weight_decay=1e-4
)

# ── Training loop ─────────────────────────────────────────────────
history = {
    "train_loss": [], "val_loss": [],
    "val_dice": [],   "val_iou": []
}

best_val_loss    = float("inf")
best_model_state = None
best_epoch       = 0
patience_counter = 0

print(f"\n{'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14} {'Val Dice':<12} {'Val IoU':<10}")
print("-" * 60)

for epoch in range(1, EPOCHS_P1 + 1):
    train_loss               = train_one_epoch(model, train_loader, optimizer_p1, DEVICE)
    val_loss, val_dice, val_iou = validate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state,
                   os.path.join(SAVE_DIR, "unetpp_resnet50_best_p1.pth"))
        marker = "  ✅ best"
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n🛑 Early stopping at epoch {epoch} (patience={PATIENCE})")
            break

    print(f"{epoch:<8} {train_loss:<14.4f} {val_loss:<14.4f} {val_dice:<12.4f} {val_iou:<10.4f}{marker}")

print(f"\n✅ Phase 1 complete. Best epoch: {best_epoch}  |  Best val loss: {best_val_loss:.4f}")
p1_history = copy.deepcopy(history)  # save phase 1 history for plotting later

## 🔓 Step 14 — Phase 2: Fine-Tuning (Unfreeze Top Encoder Layers)

**Strategy:** Unfreeze ResNet-50 layer3 + layer4 (top two blocks).  
Use very low LR (1e-5) to carefully adapt them to fluorescein images.  
**Why layer3 + layer4?** These learn high-level features (textures, shapes) — most domain-specific. Lower layers (edges, colors) are universal.

In [ ]:
# ── Load best Phase 1 weights before fine-tuning ─────────────────
model.load_state_dict(best_model_state)

# ── Unfreeze top encoder layers ───────────────────────────────────
for param in model.encoder.parameters():
    param.requires_grad = False          # Keep everything frozen first

for param in model.encoder.layer4.parameters():
    param.requires_grad = True           # Unfreeze layer4

for param in model.encoder.layer3.parameters():
    param.requires_grad = True           # Unfreeze layer3

trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Phase 2: layer3 + layer4 UNFROZEN")
print(f"  Trainable parameters: {trainable_p2:,}  (decoder + layer3 + layer4)")

optimizer_p2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P2, weight_decay=1e-4
)

# ── Fine-tuning loop ──────────────────────────────────────────────
patience_counter = 0

print(f"\n{'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14} {'Val Dice':<12} {'Val IoU':<10}")
print("-" * 60)

for epoch in range(1, EPOCHS_P2 + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer_p2, DEVICE)
    val_loss, val_dice, val_iou = validate(model, val_loader, DEVICE)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    marker = ""
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = len(p1_history["train_loss"]) + epoch
        patience_counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state,
                   os.path.join(SAVE_DIR, "unetpp_resnet50_best_final.pth"))
        marker = "  ✅ best"
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n🛑 Early stopping at fine-tune epoch {epoch}")
            break

    print(f"{epoch:<8} {train_loss:<14.4f} {val_loss:<14.4f} {val_dice:<12.4f} {val_iou:<10.4f}{marker}")

print(f"\n✅ Phase 2 complete. Best overall epoch: {best_epoch}  |  Best val loss: {best_val_loss:.4f}")

# Load best model for evaluation
model.load_state_dict(best_model_state)
print("✅ Best model weights loaded")

## 📊 Step 15 — Training Curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
p1_end       = len(p1_history["train_loss"])  # Phase 1 boundary

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("U-Net++ ResNet-50 — Training Curves", fontsize=14, fontweight="bold")

# ── Loss ──────────────────────────────────────────────────────────
axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train Loss", markersize=4)
axes[0].plot(epochs_range, history["val_loss"],   "r-o", label="Val Loss",   markersize=4)
axes[0].axvline(x=p1_end + 0.5, color="gray", linestyle="--", label="Fine-tune start")
axes[0].set_title("Combined Loss (Dice + BCE)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Dice ──────────────────────────────────────────────────────────
axes[1].plot(epochs_range, history["val_dice"], "g-o", label="Val Dice", markersize=4)
axes[1].axvline(x=p1_end + 0.5, color="gray", linestyle="--", label="Fine-tune start")
axes[1].set_title("Dice Coefficient")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ── IoU ───────────────────────────────────────────────────────────
axes[2].plot(epochs_range, history["val_iou"], "m-o", label="Val IoU", markersize=4)
axes[2].axvline(x=p1_end + 0.5, color="gray", linestyle="--", label="Fine-tune start")
axes[2].set_title("IoU (Jaccard Index)")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("IoU")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Training curves saved")

## 🧪 Step 16 — Evaluate on Test Set

In [ ]:
model.eval()

all_preds_flat = []
all_masks_flat = []
test_dices     = []
test_ious      = []

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        preds_logits  = model(images)
        preds_binary  = (torch.sigmoid(preds_logits) > 0.5).float()

        dice, iou = compute_metrics_batch(preds_logits, masks)
        test_dices.append(dice)
        test_ious.append(iou)

        all_preds_flat.append(preds_binary.cpu().numpy().flatten())
        all_masks_flat.append(masks.cpu().numpy().flatten())

all_preds_flat = np.concatenate(all_preds_flat).astype(int)
all_masks_flat = np.concatenate(all_masks_flat).astype(int)

# ── Pixel-level metrics from confusion matrix ─────────────────────
cm             = confusion_matrix(all_masks_flat, all_preds_flat, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity  = tp / (tp + fn + 1e-6)   # Recall — ulcer pixels found
specificity  = tn / (tn + fp + 1e-6)   # Non-ulcer pixels correct
precision    = tp / (tp + fp + 1e-6)   # Of predicted ulcer, how many real
accuracy     = (tp + tn) / (tp + tn + fp + fn + 1e-6)
f1           = 2 * tp / (2 * tp + fp + fn + 1e-6)
avg_dice     = float(np.mean(test_dices))
avg_iou      = float(np.mean(test_ious))

print("" + "=" * 50)
print("  TEST SET RESULTS — U-Net++ ResNet-50")
print("=" * 50)
print(f"  TP: {tp:,}   TN: {tn:,}   FP: {fp:,}   FN: {fn:,}")
print("-" * 50)
print(f"  Dice Coefficient : {avg_dice:.4f}")
print(f"  IoU (Jaccard)    : {avg_iou:.4f}")
print(f"  Sensitivity      : {sensitivity:.4f}  (recall — ulcer detection rate)")
print(f"  Specificity      : {specificity:.4f}")
print(f"  Precision        : {precision:.4f}")
print(f"  Accuracy         : {accuracy:.4f}")
print(f"  F1 Score         : {f1:.4f}")
print("=" * 50)

# ── Save metrics ──────────────────────────────────────────────────
metrics = {
    "model"          : "UNet++ ResNet-50",
    "dice"           : round(avg_dice,    4),
    "iou"            : round(avg_iou,     4),
    "sensitivity"    : round(sensitivity, 4),
    "specificity"    : round(specificity, 4),
    "precision"      : round(precision,   4),
    "accuracy"       : round(accuracy,    4),
    "f1_score"       : round(f1,          4),
    "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn)
}
with open(os.path.join(SAVE_DIR, "test_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)
print("✅ Metrics saved to Drive")

## 📊 Step 17 — Confusion Matrix Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Background", "Ulcer"]
).plot(ax=ax, colorbar=False, cmap="Blues")

ax.set_title(
    f"U-Net++ ResNet-50 — Test Set Confusion Matrix\n"
    f"Sensitivity: {sensitivity:.4f}  |  Specificity: {specificity:.4f}  |  "
    f"F1: {f1:.4f}  |  Dice: {avg_dice:.4f}",
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved")

## 🖼️ Step 18 — Prediction Visualization (Image | GT Mask | Predicted Mask)

In [ ]:
model.eval()

# Get one batch from test loader
images_batch, masks_batch = next(iter(test_loader))
images_batch = images_batch.to(DEVICE)

with torch.no_grad():
    preds_logits = model(images_batch)
    preds_binary = (torch.sigmoid(preds_logits) > 0.5).float()

n_show = min(6, len(images_batch))
fig, axes = plt.subplots(n_show, 3, figsize=(12, n_show * 4))
fig.suptitle("U-Net++ ResNet-50 — Predictions vs Ground Truth",
             fontsize=14, fontweight="bold")

for i in range(n_show):
    img_np   = denormalize(images_batch[i])
    mask_np  = masks_batch[i].squeeze().cpu().numpy()
    pred_np  = preds_binary[i].squeeze().cpu().numpy()

    smooth  = 1e-6
    inter   = (pred_np * mask_np).sum()
    dice_i  = (2 * inter + smooth) / (pred_np.sum() + mask_np.sum() + smooth)
    iou_i   = (inter + smooth) / (pred_np.sum() + mask_np.sum() - inter + smooth)

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title("Raw Fluorescein Image", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title(f"Ground Truth\n({int(mask_np.sum())} ulcer px)", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred_np, cmap="gray")
    axes[i, 2].set_title(
        f"Prediction ({int(pred_np.sum())} px)\nDice: {dice_i:.4f}  IoU: {iou_i:.4f}",
        fontsize=9
    )
    axes[i, 2].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "predictions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Prediction visualization saved")

## 🌡️ Step 19 — Grad-CAM Explainability

Grad-CAM shows **which image regions most influenced the prediction**.  
This is critical for clinical acceptance — doctors need to trust the model.

In [ ]:
import cv2

class GradCAM:
    """
    Grad-CAM for segmentation models.
    Hooks into the target layer, computes gradient-weighted activation maps.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.gradients    = None
        self.activations  = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor):
        """
        input_tensor: (1, 3, H, W) on DEVICE
        Returns: heatmap (H, W) numpy array in [0, 1]
        """
        self.model.eval()
        input_tensor = input_tensor.requires_grad_(True)

        output = self.model(input_tensor)          # (1, 1, H, W)
        score  = torch.sigmoid(output).mean()       # Scalar — overall ulcer probability
        self.model.zero_grad()
        score.backward()

        # GAP over spatial dims
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam     = (weights * self.activations).sum(dim=1).squeeze()  # (H, W)
        cam     = torch.relu(cam).cpu().numpy()

        # Resize to input image size
        cam = cv2.resize(cam, (input_tensor.shape[3], input_tensor.shape[2]))

        # Normalize to [0, 1]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


# ── Attach to encoder layer4 (deepest encoder features) ──────────
grad_cam = GradCAM(model, target_layer=model.encoder.layer4)

# ── Generate and visualize for 3 test images ─────────────────────
n_cam = 3
fig, axes = plt.subplots(n_cam, 4, figsize=(16, n_cam * 4))
fig.suptitle("Grad-CAM Explainability — U-Net++ ResNet-50",
             fontsize=14, fontweight="bold")

for i in range(n_cam):
    img_tensor, mask_tensor = test_dataset[i]
    input_t   = img_tensor.unsqueeze(0).to(DEVICE)  # (1, 3, H, W)

    # Prediction
    with torch.no_grad():
        pred_logit = model(input_t)
        pred_bin   = (torch.sigmoid(pred_logit) > 0.5).float().squeeze().cpu().numpy()

    # Grad-CAM heatmap
    heatmap = grad_cam.generate(input_t)

    img_np   = denormalize(img_tensor)
    mask_np  = mask_tensor.squeeze().numpy()

    # Overlay heatmap on original image
    heatmap_color = cv2.applyColorMap(
        (heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    overlay       = 0.5 * img_np + 0.5 * heatmap_color
    overlay       = np.clip(overlay, 0, 1)

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title("Input Image", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred_bin, cmap="gray")
    axes[i, 2].set_title("Prediction", fontsize=9)
    axes[i, 2].axis("off")

    axes[i, 3].imshow(overlay)
    axes[i, 3].set_title("Grad-CAM Overlay\n(red = high attention)", fontsize=9)
    axes[i, 3].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "gradcam.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Grad-CAM visualization saved")

## 💾 Step 20 — Export Model for FastAPI

Save in two formats:
- `.pth` — PyTorch weights (primary, used in FastAPI backend)
- TorchScript `.pt` — optional, for production inference without source code

In [ ]:
# ── 1. Save best model weights (.pth) ─────────────────────────────
pth_path = os.path.join(SAVE_DIR, "unetpp_resnet50_best_final.pth")
torch.save(best_model_state, pth_path)
print(f"✅ PyTorch weights saved : {pth_path}")

# ── 2. Save full model info for FastAPI ───────────────────────────
model_info = {
    "model_type"      : "UnetPlusPlus",
    "encoder"         : "resnet50",
    "encoder_weights" : "imagenet",
    "in_channels"     : 3,
    "classes"         : 1,
    "img_size"        : IMG_SIZE,
    "imagenet_mean"   : IMAGENET_MEAN,
    "imagenet_std"    : IMAGENET_STD,
    "threshold"       : 0.5,
    "weights_file"    : "unetpp_resnet50_best_final.pth"
}
with open(os.path.join(SAVE_DIR, "model_info.json"), "w") as f:
    json.dump(model_info, f, indent=4)
print(f"✅ Model info JSON saved")

# ── 3. TorchScript export (optional, for faster inference) ────────
try:
    model.eval()
    dummy_input  = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    scripted     = torch.jit.trace(model, dummy_input)
    script_path  = os.path.join(SAVE_DIR, "unetpp_resnet50_scripted.pt")
    scripted.save(script_path)
    print(f"✅ TorchScript model saved: {script_path}")
except Exception as e:
    print(f"ℹ️  TorchScript export skipped: {e}")

# ── 4. Save full history & metrics ────────────────────────────────
with open(os.path.join(SAVE_DIR, "training_history.json"), "w") as f:
    json.dump({
        "train_loss": history["train_loss"],
        "val_loss"  : history["val_loss"],
        "val_dice"  : history["val_dice"],
        "val_iou"   : history["val_iou"],
        "best_epoch": best_epoch,
        "test_metrics": metrics
    }, f, indent=4)
print(f"✅ Training history saved")

print(f"\n" + "=" * 55)
print(f"  ALL FILES SAVED TO: {SAVE_DIR}")
print("=" * 55)
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:<45} {size/1024:.1f} KB")

## 🔌 Step 21 — FastAPI Inference Snippet (Reference)

Copy this into your FastAPI `main.py` for Phase 2.

In [ ]:
fastapi_code = '''
# ── FastAPI backend — save as backend/main.py ─────────────────────

import io
import json
import numpy as np
from PIL import Image
import torch
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Corneal Ulcer Detection API")

# Allow React frontend (Vercel) to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Load model once on startup ────────────────────────────────────
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 256

model = smp.UnetPlusPlus(
    encoder_name="resnet50",
    encoder_weights=None,
    in_channels=3,
    classes=1,
    activation=None
).to(DEVICE)

model.load_state_dict(torch.load("unetpp_resnet50_best_final.pth",
                                  map_location=DEVICE))
model.eval()

transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
    ToTensorV2()
])

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    # Read and preprocess image
    contents = await file.read()
    image    = np.array(Image.open(io.BytesIO(contents)).convert("RGB"))
    tensor   = transform(image=image)["image"].unsqueeze(0).to(DEVICE)

    # Inference
    with torch.no_grad():
        output      = model(tensor)
        probability = torch.sigmoid(output).mean().item()   # Overall ulcer probability
        pred_mask   = (torch.sigmoid(output) > 0.5).float().squeeze().cpu().numpy()

    ulcer_pixel_ratio = float(pred_mask.sum()) / (IMG_SIZE * IMG_SIZE)

    return {
        "prediction"         : "Ulcer Detected" if probability > 0.3 else "No Ulcer",
        "confidence"         : round(probability * 100, 2),
        "ulcer_area_percent" : round(ulcer_pixel_ratio * 100, 2)
    }

@app.get("/health")
def health():
    return {"status": "ok", "model": "UNet++ ResNet-50"}
'''

print(fastapi_code)

# Also save to Drive for easy copy
with open(os.path.join(SAVE_DIR, "fastapi_main_reference.py"), "w") as f:
    f.write(fastapi_code.strip())
print("\n✅ FastAPI reference code saved to Drive")

## ✅ Step 22 — Final Summary

Run this cell at the end to print everything cleanly for your viva.

In [ ]:
print("\n" + "=" * 60)
print("  FINAL SUMMARY — U-Net++ ResNet-50")
print("=" * 60)
print(f"  Dataset    : SUSTech-SYSU ({len(image_paths)} fluorescein images)")
print(f"  Split      : 70 / 15 / 15  (train/val/test)")
print(f"  Input size : {IMG_SIZE}×{IMG_SIZE} RGB")
print()
print(f"  Phase 1    : Encoder frozen   | lr={LR_P1}")
print(f"  Phase 2    : layer3+4 unfrozen | lr={LR_P2}")
print(f"  Best epoch : {best_epoch}")
print()
print("  TEST METRICS")
print(f"  {'Dice':<16}: {metrics['dice']}")
print(f"  {'IoU':<16}: {metrics['iou']}")
print(f"  {'Sensitivity':<16}: {metrics['sensitivity']}")
print(f"  {'Specificity':<16}: {metrics['specificity']}")
print(f"  {'Precision':<16}: {metrics['precision']}")
print(f"  {'F1 Score':<16}: {metrics['f1_score']}")
print(f"  {'Accuracy':<16}: {metrics['accuracy']}")
print()
print("  SAVED FILES")
for fname in sorted(os.listdir(SAVE_DIR)):
    print(f"    {fname}")
print("=" * 60)